# v2 四频 TCN 平台测试

上传本 Notebook 和同目录的 `v2-model.json`，共两个 UTF-8 文本文件。
运行 `v2-20261011-s42`，按验证集选定第 4 轮权重。本地判定：**未达到本地改善条件**。导出理由：用户要求将 v2 重新打包为尽量少的纯文本附件，用于平台测试；保留本地未超过 v1 的判定。。
JSON 包含完整数值权重、四频统计量、配置、依赖及训练／推理源码；加载后按原 dtype 恢复，不截断小数精度。
入口为 `main(datasources, start_date, end_date)`。导入不查询行情；推理仅使用授权股票池和四频行情，不读取标签或训练。
推理依赖：numpy>=1.26,<3, pandas>=2.2,<3, pyarrow>=14, torch>=2.6,<3，以及平台 `dai`。有 CUDA 时使用 GPU，否则使用 CPU；平台环境与成绩须实际验证。
训练复现：先安装 JSON 所附 `requirements-training.txt` 对应依赖，提供授权原始 Parquet，再显式调用 `reproduce("reproduced-v2", "/path/to/data")`。
复现工作目录必须不存在；该调用会生成预处理数据并运行正式训练。仅执行下方调试单元格不会训练。
本地评估摘要与原权重 SHA-256 见 JSON 的 `metadata`；逐文件哈希及本地入口验证记录位于仓库 `experiments/v2-20261011-s42-submission.json`，无需上传。


In [ ]:
"""两文件提交的 Notebook 入口模板；导入时不读取数据、不创建附件。"""

ARTIFACT_SHA256 = "6d3ff72a3db357f0005e6f94c893d2e44d48ea74481ed269508dfa673453f6e7"
_RUNTIME = None


def _bundle_directory():
    from pathlib import Path
    return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def _read_bundle():
    import hashlib
    import json
    raw = (_bundle_directory() / "v2-model.json").read_bytes()
    if hashlib.sha256(raw).hexdigest() != ARTIFACT_SHA256:
        raise ValueError("v2-model.json 与 Notebook 不匹配")
    return json.loads(raw.decode("utf-8"))


def _load_modules(bundle, paths):
    import torch
    import builtins
    import hashlib
    from pathlib import Path
    from types import ModuleType
    modules = {}

    def local_import(name, globals=None, locals=None, fromlist=(), level=0):
        if level == 0 and name in modules:
            return modules[name]
        return builtins.__import__(name, globals, locals, fromlist, level)

    for relative in paths:
        entry = bundle["sources"][relative]
        if hashlib.sha256(entry["text"].encode("utf-8")).hexdigest() != entry["sha256"]:
            raise ValueError("内嵌源码哈希不匹配: " + relative)
        name = Path(relative).stem
        module = ModuleType(name)
        module.__file__ = str(_bundle_directory() / Path(relative).name)
        module.__dict__["__builtins__"] = dict(vars(builtins), __import__=local_import)
        modules[name] = module
        exec(compile(entry["text"], module.__file__, "exec"), module.__dict__)
    return modules


def _runtime():
    global _RUNTIME
    if _RUNTIME is None:
        bundle = _read_bundle()
        modules = _load_modules(bundle, bundle["runtime_sources"])
        _RUNTIME = bundle, modules
    return _RUNTIME


def main(datasources, start_date, end_date):
    bundle, modules = _runtime()
    return modules["v2_predict"].predict(
        datasources, start_date, end_date, model_dir=_bundle_directory(), bundle=bundle,
    )


def reproduce(workspace, data_root):
    """显式调用才启动预处理和训练；workspace 必须是不存在的新目录。"""
    bundle = _read_bundle()
    modules = _load_modules(bundle, ["solutions/v2/v2_reproduce.py"])
    return modules["v2_reproduce"].reproduce(bundle, workspace, data_root)


In [ ]:
if __name__ == "__main__":
    result = main(
        {f: "cpt_jyc_2026_stock_" + f for f in ["bar1m", "bar5m", "bar15m", "bar30m"]},
        "2024-12-30", "2024-12-31",
    )
    import numpy as np
    assert list(result.columns) == ["date", "instrument", "factor"]
    assert not result.empty and result["factor"].dtype == np.dtype("float64")
    assert not result.duplicated(["date", "instrument"]).any()
    assert np.isfinite(result["factor"]).all()
    print("输出行数：", len(result))
    print(result.dtypes)
    display(result.head())
